# Stability Analysis - Demo_IEEE14 Package Parameter Sweep

Start from the `Demo_IEEE14` package model, change one parameter, and compare the modes across the sweep. This is the package-model counterpart of the single-file `StabilityAnalysis_BESS` notebook.

Set `REINITIALIZE_EACH_CASE` to choose whether the package is initialized once before the sweep, or reinitialized after each parameter update.

In [1]:
include("../scripts/dictionaries.jl")
include("../scripts/helpers.jl")
include("../scripts/parametric_study_helpers.jl")
include("../scripts/package_model_initialization.jl")
include("scripts/linearize_scripting.jl")

using .WorkflowHelpers
using .ParametricStudyHelpers
using .PackageModelInitialization
using OMJulia, LinearAlgebra, DataFrames

## User Configuration


In [2]:
MODEL = "Demo_IEEE14.IEEE14DisconnectLine"
SOURCE_PACKAGE = split(MODEL, ".")[1]
MODEL_DIR = abspath(joinpath("models", SOURCE_PACKAGE))
MODELS_PKG_PATH = joinpath(MODEL_DIR, "package.mo")

OMLIB_DIR = joinpath(homedir(), ".openmodelica", "libraries")
ENV["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = joinpath(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")
DYNAWO_PKG_PATH = abspath("../dynawo_library/Dynawo/package.mo")

SWEEP_COMPONENT = "LineB2B5"
SWEEP_PARAMETER = "XPu"
SWEEP_VALUES = [0.12, 0.173884267, 0.24]

REINITIALIZE_EACH_CASE = false

LINEARIZATION_TIME = "2.0"
MODE_TOL = 1e-8

INIT_MODEL_BY_COMPONENT = Dict{String, String}()

# Leave empty to disable slack-specific handling.
SLACK_COMPONENT = "Gen1"

OUTPUT_DIR = abspath("outputs/Demo_IEEE14_XPu_stability_sweep");

## Prepare OpenModelica Session


In [3]:
BUILD_OUTPUT_DIR = joinpath(OUTPUT_DIR, "omc-build")

isfile(MODELS_PKG_PATH) || error("Package file not found: $MODELS_PKG_PATH")
isfile(DYNAWO_PKG_PATH) || error("Dynawo package not found: $DYNAWO_PKG_PATH")
isfile(MODELICA_PKG_PATH) || error("Modelica package not found: $MODELICA_PKG_PATH")

rm(OUTPUT_DIR; recursive = true, force = true)
mkpath(BUILD_OUTPUT_DIR)

StudyOMC = OMJulia.OMCSession()
load_modelica_file!(
    StudyOMC,
    MODELS_PKG_PATH,
    MODELICA_PKG_PATH,
    DYNAWO_PKG_PATH,
)
omc_call(StudyOMC, "checkModel($MODEL)", parsed = false)

MODEL_CHAIN = get_inheritance_chain(StudyOMC, MODEL)

parameter_models = String[]
for model in MODEL_CHAIN
    model_name = String(model)
    if haskey(get_all_components(StudyOMC, model_name), SWEEP_COMPONENT)
        push!(parameter_models, model_name)
    end
end

isempty(parameter_models) && error("Component $SWEEP_COMPONENT was not found in the inheritance chain for $MODEL")
PARAMETER_MODEL = first(parameter_models)
# (sin OMJulia.quit aqui: reutilizamos StudyOMC en la celda siguiente)

println("Package model checked successfully.")

[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.dOChJsWFJJ"


Package model checked successfully.


## Initialize the Base Package

When `REINITIALIZE_EACH_CASE` is `false`, initialize the base package once here and reuse it for every sweep case. When it is `true`, this cell only closes the session opened above.

In [4]:
if !REINITIALIZE_EACH_CASE
    base_initialized_case = initialize_loaded_package_model(
        StudyOMC;
        source_model = MODEL,
        model_chain = MODEL_CHAIN,
        case_name = SOURCE_PACKAGE * "_base",
        output_dir = OUTPUT_DIR,
        modelica_package_path = MODELICA_PKG_PATH,
        dynawo_package_path = DYNAWO_PKG_PATH,
        init_model_by_component = INIT_MODEL_BY_COMPONENT,
        slack_component = SLACK_COMPONENT,
    )

    base_initialized_package_file = base_initialized_case.initialized_package_file
    base_initialized_model = base_initialized_case.initialized_model

    base_initialized_model_map = initialized_name_map(
        MODEL_CHAIN,
        base_initialized_case.initialized_package,
    )
    base_parameter_model = base_initialized_model_map[PARAMETER_MODEL]
    println("Base package initialized once: ", base_initialized_model)
end

OMJulia.quit(StudyOMC)

[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.iiYiHtmaCC"


Base package initialized once: Demo_IEEE14_base_initialized.IEEE14DisconnectLine_initialized


## Run the Parameter Sweep


In [5]:
A_matrices = Dict{Float64, Matrix{Float64}}()
linear_states_by_case = Dict{Float64, Vector{String}}()
linearization_sizes = NamedTuple[]

for value in SWEEP_VALUES
    label = case_label(value)
    parameter_name = replace(SWEEP_PARAMETER, "." => "_")
    case_model = join([SOURCE_PACKAGE, SWEEP_COMPONENT, parameter_name, label], "_")
    case_build_dir = joinpath(BUILD_OUTPUT_DIR, case_model)
    mkpath(case_build_dir)

    StudyOMC = OMJulia.OMCSession()

    try
        if REINITIALIZE_EACH_CASE
            load_modelica_file!(
                StudyOMC,
                MODELS_PKG_PATH,
                MODELICA_PKG_PATH,
                DYNAWO_PKG_PATH,
            )

            omc_call(
                StudyOMC,
                "setParameterValue($(PARAMETER_MODEL), $(SWEEP_COMPONENT).$(SWEEP_PARAMETER), $(string(value)))",
                parsed = false,
            )

            initialized_case = initialize_loaded_package_model(
                StudyOMC;
                source_model = MODEL,
                model_chain = MODEL_CHAIN,
                case_name = case_model,
                output_dir = OUTPUT_DIR,
                modelica_package_path = MODELICA_PKG_PATH,
                dynawo_package_path = DYNAWO_PKG_PATH,
                init_model_by_component = INIT_MODEL_BY_COMPONENT,
                slack_component = SLACK_COMPONENT,
            )

            linearized_model = initialized_case.initialized_model
        else
            load_modelica_file!(
                StudyOMC,
                base_initialized_package_file,
                MODELICA_PKG_PATH,
                DYNAWO_PKG_PATH,
            )

            omc_call(
                StudyOMC,
                "setParameterValue($(base_parameter_model), $(SWEEP_COMPONENT).$(SWEEP_PARAMETER), $(string(value)))",
                parsed = false,
            )

            linearized_model = base_initialized_model
        end

        case_simflags = simulation_flags_without_log_stats(StudyOMC, linearized_model)
        lin = linearize_scripting(StudyOMC, linearized_model;
            startTime = 0.0, stopTime = LINEARIZATION_TIME, stepSize = 0.001, tolerance = 1e-6,
            simflags = case_simflags, outdir = case_build_dir)

        A_matrices[value] = lin.A
        linear_states_by_case[value] = lin.states

        if value == first(SWEEP_VALUES)
            push!(linearization_sizes, (
                A = join(string.(size(lin.A)), "x"),
                B = join(string.(size(lin.B)), "x"),
                C = join(string.(size(lin.C)), "x"),
                D = join(string.(size(lin.D)), "x"),
            ))
        end
    finally
        OMJulia.quit(StudyOMC)
    end

    println("Finished $SWEEP_COMPONENT.$SWEEP_PARAMETER = $value")
end

linear_states = linear_states_by_case[first(SWEEP_VALUES)]

for value in SWEEP_VALUES
    if linear_states_by_case[value] != linear_states
        error("Linear state order changed for $SWEEP_COMPONENT.$SWEEP_PARAMETER = $value")
    end
end

println("Linearization sweep finished.")

[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.AcKvpcHgm8"


LoadError: Notification: Modelica requested package Complex of version 3.2.3. Complex 4.1.0 is used instead which states that it is fully compatible without conversion script needed.
Notification: Modelica requested package ModelicaServices of version 3.2.3. ModelicaServices 4.1.0 is used instead which states that it is fully compatible without conversion script needed.
[/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/Electrical/Controls/Basics/SwitchOff/SwitchOffLogic.mo:20:3-20:98:writable] Warning: Connector switchOffSignal1 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/Electrical/Controls/Basics/SwitchOff/SwitchOffLogic.mo:21:3-21:125:writable] Warning: Connector switchOffSignal2 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[/home/clarafercas/Model_library/dyn

## Linearization Summary


In [ ]:
sizes = linearization_sizes[1]

linearization_summary = DataFrame([(
    n_states = length(linear_states),
    size_A = sizes.A,
    size_B = sizes.B,
    size_C = sizes.C,
    size_D = sizes.D,
)])

display(linearization_summary)


## Analyze Modes


In [ ]:
mode_rows = NamedTuple[]
summary_rows = NamedTuple[]

for value in SWEEP_VALUES
    A = A_matrices[value]

    eigenvalues = eigvals(A)
    real_parts = real.(eigenvalues)

    for (mode_index, lambda) in enumerate(eigenvalues)
        real_part = real(lambda)
        imag_part = imag(lambda)

        stability = if real_part > MODE_TOL
            "unstable"
        elseif real_part < -MODE_TOL
            "stable"
        else
            "marginal"
        end

        push!(mode_rows, (
            parameter_value = value,
            mode = mode_index,
            real_part = real_part,
            imag_part = imag_part,
            stability = stability,
        ))
    end

    push!(summary_rows, (
        parameter_value = value,
        max_real_part = maximum(real_parts),
        unstable_modes = count(real_parts .> MODE_TOL),
        marginal_modes = count(abs.(real_parts) .<= MODE_TOL),
        stable_modes = count(real_parts .< -MODE_TOL),
    ))
end

mode_df = sort(DataFrame(mode_rows), [:parameter_value, :mode])
stability_summary = sort(DataFrame(summary_rows), :parameter_value)

println("Mode analysis finished.")


## Check Unstable Modes


In [ ]:
unstable_modes = Int[]

for mode in sort(unique(mode_df.mode))
    rows_for_mode = mode_df[mode_df.mode .== mode, :]

    if any(rows_for_mode.stability .== "unstable")
        push!(unstable_modes, mode)
    end
end

unstable_mode_table = DataFrame(mode = unstable_modes)

for value in SWEEP_VALUES
    column_name = Symbol("$SWEEP_PARAMETER=$(value)")
    stability_values = String[]

    for mode in unstable_modes
        rows_for_mode = mode_df[(mode_df.mode .== mode) .& (mode_df.parameter_value .== value), :]
        push!(stability_values, rows_for_mode.stability[1])
    end

    unstable_mode_table[!, column_name] = stability_values
end

unstable_mode_rows = in.(mode_df.mode, Ref(unstable_modes))
unstable_mode_df = sort(mode_df[unstable_mode_rows, :], [:mode, :parameter_value])

println("Unstable-mode check finished.")


## Modes


In [ ]:
println("Stability summary")
display(stability_summary)

println("Modes that become unstable")
display(unstable_mode_table)


## Mode Plots


In [ ]:
using Plots

if isempty(unstable_modes)
    println("No modes become unstable for these sweep values.")
else
    # symmetric limits centred on the origin so all four quadrants are visible
    r_lim = 1.1 * maximum(abs.(unstable_mode_df.real_part))
    i_lim = 1.1 * maximum(abs.(unstable_mode_df.imag_part))

    locus_plot = plot(
        xlabel = "real(lambda)",
        ylabel = "imag(lambda)",
        title = "Eigenvalue evolution vs $SWEEP_COMPONENT.$SWEEP_PARAMETER",
        legend = :outertopright,
        xlims = (-r_lim, r_lim),
        ylims = (-i_lim, i_lim),
    )
    vline!(locus_plot, [0.0], line = (:dash, :black), label = "")  # Re = 0  (stability boundary)
    hline!(locus_plot, [0.0], line = (:dash, :black), label = "")  # Im = 0

    for mode in unstable_modes
        mode_data = unstable_mode_df[unstable_mode_df.mode .== mode, :]
        plot!(
            locus_plot,
            mode_data.real_part,
            mode_data.imag_part,
            marker = :circle,
            label = "mode $mode",
            series_annotations = text.(string.(mode_data.parameter_value), 8, :bottom),
        )
    end

    display(locus_plot)
end
